# Run-book 198 — does mpc-fast-extract make datacube creation faster?

**What this proves:** for the same geometries, dates and bands, how long rise-eo takes to create
datacubes (download whole granules, then build) against mpc-fast-extract (stream only the blocks
it needs into a Zarr, then cut that into rise-eo cubes). It also says how closely the two sets of
cubes agree, pixel by pixel. It is the P0 spike of issue #198: the numbers decide whether an
mpc-fast-extract build backend gets a spec. Two workloads:
- **fields**: the 900 EuroCrops fields of `AT_2018_TRAIN` (the training case: many tiny cubes);
- **cells**: `AT_ROI` cut into grid cells (the inference case: a few hundred large cubes).

**What it does not show:** model results, CDSE, the AML cluster (both sides run on one machine),
ROIs across UTM zones (all of Austria is UTM 33N).

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, or a failing cell's message. Every check is a plain `assert` you can read: the
text above a check cell says what PASS means, and the `assert` says the same thing in code.

**Needs** (all estimates):
- your AML compute instance: Linux x86-64 with AVX2 (any Dv5), in West Europe next to the imagery;
- network to Microsoft Planetary Computer (anonymous, no credentials);
- disk: about 103 GB for the archive (measured on the first run) plus about 12 GB for the Zarrs, the
  cubes and Step 11, so about 120 GB free on a local disk;
- time: 30–45 min of one-time setup, then about 1–1.5 h of running, mostly the download and the
  rise-eo builds.

<!-- Rules for writing one (spec 24 amendment A1):
  - call rise-eo's public functions directly, keyword arguments spelled out;
  - small helpers only (~15 lines), defined just above their first use, with a docstring;
  - no environment variables, no try/except around a whole step, no result-file plumbing;
  - every step reads what it needs from disk + Settings, never from an earlier step's variables;
  - a plan cell (counts + GB, no transfer) before any step that moves more than ~1 GB;
  - a deliberate kill runs in a child process (os._exit would kill this kernel). -->

## Before you start

**One-time setup on the compute instance.** Open a terminal on it (Azure ML studio → Compute →
your compute instance → Terminal) and run these. They install rise-eo at this branch, Rust, and
mpc-fast-extract into rise-eo's `.venv`. Set `REPOS` to the folder that will hold both checkouts:
Settings expects `mpc-fast-extract` next to `rise-eo`. Keep it on the instance's own disk;
`~/cloudfiles` is a network share.

```bash
REPOS="$HOME"   # the folder that holds both checkouts

# 1. rise-eo at this branch, in a Python 3.11 venv (the compute instance ships conda)
conda create -y -p "$REPOS/py311" python=3.11
git clone https://github.com/nasaharvest/rise-eo.git "$REPOS/rise-eo"
cd "$REPOS/rise-eo" && git checkout spike/p0-mpc-fast-extract-parity
"$REPOS/py311/bin/python" -m venv .venv
.venv/bin/pip install -e ".[dev,local,s3,notebooks,grid,azure,aml,mpc,titiler,serving]" ipykernel

# 2. Rust (under $HOME/.cargo), then mpc-fast-extract at the commit this run-book was written against
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
. "$HOME/.cargo/env"   # puts cargo on PATH for this shell; the pip build below needs it
git clone https://github.com/rohansaw/mpc-fast-extract.git "$REPOS/mpc-fast-extract"
git -C "$REPOS/mpc-fast-extract" checkout da6bf12
.venv/bin/pip install -e "$REPOS/mpc-fast-extract/python[raster]"   # compiles the engine: ~5-15 min

# 3. find a local disk with ~120 GB free for OUT in Settings (often /mnt); not ~/cloudfiles
df -h
# /mnt belongs to root: create OUT there once and make it yours
sudo mkdir -p /mnt/rise-p0 && sudo chown "$USER" /mnt/rise-p0
```

If the mpc-fast-extract clone asks for a login, the repository is private: ask Rohan for access.
If `pip` reports a version conflict, paste its message back instead of forcing it.

**Kernel — which Python runs the cells.** In Azure ML studio, open the compute instance in
**VS Code (Desktop)**, open the folder `$REPOS/rise-eo`, open this notebook, then **Select Kernel** →
**Python Environments** → **`.venv`**. The **Setup** cell below checks this and stops with a
message if the kernel is wrong.

**Code — which `rise-eo` gets imported.** The Setup cell puts **this notebook's own** `src/` first
on Python's import path and checks it worked.

**Stop / resume.** Interrupt the kernel (■) to stop. After a kernel restart, run **Setup**,
**Settings** and **Helpers** again, then continue at any step: every step reads what it needs from
disk. A timed step that already finished is skipped (its time is in `OUT/timings/`). To time a step
again, delete its file there. The download resumes where it stopped, but then its time covers only
the rest: for a clean time, delete `OUT/archive` and its timing file, and run Step 4 again.

**When you are done.** Paste back what the **Summary** cell prints (plus any failing cell's
message). Run the Summary even if a check failed: it reads whatever is on disk. Then
**Restart & Clear All Outputs** and save: this file is tracked in git, and a test refuses saved
outputs.

In [ ]:
# Setup -- which Python, and which rise-eo. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be rise-eo's own virtual environment, fsd/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not rise-eo's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the rise-eo checkout that owns the venv

# 2. Which rise-eo? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "rise_eo").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import rise_eo

assert Path(rise_eo.__file__).is_relative_to(CHECKOUT / "src"), (
    f"rise-eo was already imported from {rise_eo.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("rise-eo    :", rise_eo.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else. `mpc_config` is an input too:
it is the mpc-fast-extract config that mirrors rise-eo's build, with a comment on each choice.

In [ ]:
# Settings
import datetime
import math
import os

import pandas as pd

from rise_eo import collections as rise_collections
from rise_eo.api import archive_catalog_filepath

OUT = Path("/mnt/rise-p0")   # a LOCAL disk of the compute instance (`df -h`), never ~/cloudfiles
NEED_GB = 120                # free at OUT before the download: ~103 GB archive + ~12 GB Zarrs, cubes, Step 11
MPC_FAST_EXTRACT = CHECKOUT.parent / "mpc-fast-extract"   # next to rise-eo; pip-installed into .venv
MPC_FAST_EXTRACT_COMMIT = "da6bf12"                    # the version this run-book was written against

SHAPES = CHECKOUT / "notebooks" / "shapefiles"
LABELS = SHAPES / "AT_2018_TRAIN.geojson"   # 900 EuroCrops fields: the training case
ROI = SHAPES / "AT_ROI.geojson"             # 4 MGRS tiles, cut into grid cells: the inference case
WORKLOADS = {"fields": "field_id", "cells": "id"}   # workload -> its id column

STARTDATE = datetime.datetime(2018, 4, 1)
ENDDATE = datetime.datetime(2018, 10, 1)
MOSAIC_DAYS = 20                            # 10 calendar windows
WINDOWS = pd.date_range(STARTDATE, freq=f"{MOSAIC_DAYS}D",
                        periods=math.ceil((ENDDATE - STARTDATE).days / MOSAIC_DAYS))
BANDS = ["B04", "B08", "SCL"]               # what rise-eo downloads and builds from
CUBE_BANDS = ["B04", "B08"]                 # what a cube holds once SCL has masked it
COLLECTION = "sentinel-2-l2a"
PROCESSING = "latest"
SCL_CLASSES = list(rise_collections.get(COLLECTION).mask_spec.classes)   # rise-eo's own mask

MAX_TILES = 300          # download guard: Apr-Sep over 4 tiles is about 184 granules
MAX_CONCURRENT = 32      # band files in flight during the download (MPC is in the same region)
CORES = max(1, os.cpu_count() // 2)   # parallel rise-eo builds; each holds a few GB (estimate)
GRID_CRS = "EPSG:32633"  # every Austria tile is in UTM 33N, the zone rise-eo builds in here
RES = 10                 # metres, rise-eo's reference band (B08)
MIN_AGREEMENT = 0.99     # PASS bar for the pixel checks in Step 8: a first guess, not a spec
CORES_FAIR = os.cpu_count()   # Step 9: one rise-eo build per core; mpc-fast-extract uses every core

SHAPES_OUT, TIMINGS, ARCHIVE = OUT / "shapes", OUT / "timings", OUT / "archive"
CATALOG = archive_catalog_filepath(str(ARCHIVE), COLLECTION)
EXPORT = MAIN_CHECKOUT / "tests" / "outputs" / "198_p0_export"   # Step 10: home disk, survives a stop

# Step 11: one single-tile cell in June 2023, where every granule carries the -1000 DN offset
OFFSET_ROI = SHAPES / "s2grid=4772924.geojson"   # inside T33UWP: no tile-overlap tie-break
OFFSET_START, OFFSET_END = datetime.datetime(2023, 6, 1), datetime.datetime(2023, 7, 1)
OFFSET_MOSAIC_DAYS = 10                          # 3 windows
OFFSET_WINDOWS = pd.date_range(OFFSET_START, freq=f"{OFFSET_MOSAIC_DAYS}D",
                               periods=math.ceil((OFFSET_END - OFFSET_START).days / OFFSET_MOSAIC_DAYS))
OFFSET_BANDS = ["B04", "B08", "B8A", "SCL"]      # B8A is 20 m: both tools resample it to 10 m
OFFSET_CUBE_BANDS = ["B04", "B08", "B8A"]
OFFSET_MAX_TILES = 60                            # download guard: one tile x one month
OFFSET_RUN = OUT / "offset_2023"


def mpc_config(workload):
    """The mpc-fast-extract config that mirrors rise-eo's build for one workload."""
    return {
        "geometries": str(SHAPES_OUT / f"{workload}.geojson"),
        "id_field": WORKLOADS[workload],
        "start": STARTDATE.date().isoformat(),
        "end": (ENDDATE - datetime.timedelta(days=1)).date().isoformat(),  # inclusive; rise-eo
                                                # stops the day before ENDDATE
        "period": f"{MOSAIC_DAYS}d",            # windows anchored at start, like rise-eo's calendar
        "all_touched": True,                    # rise-eo crops with all_touched=True
        "sentinel2": {
            "bands": CUBE_BANDS,
            "composite": "median",              # rise-eo's only mosaic method
            "fuse": "first_valid",              # rise-eo merges one day's granules first-valid
            "cloud": {"scl_mask": SCL_CLASSES,
                      "max_scene_cloud_cover": 100},   # rise-eo drops no scene for cloud cover
        },
        "grid": {"crs": GRID_CRS, "resolution": RES},  # defaults: nearest, pixel edges on 10 m
        "output": {"path": str(OUT / "mpc" / workload / "engine"), "stats": False,
                   "rasters": {"format": "zarr",
                               "dtype": "float32"}},    # float32: DN comes back exactly (x 1e4)
    }

## Helpers

Small functions several steps use. Run this cell after Settings, and again after a restart.

In [ ]:
# Helpers
import json
import time

import geopandas as gpd
import mpc_fast_extract as mfe
import numpy as np


def timed(name, fn):
    """Run fn() unless TIMINGS/<name>.json exists; save its wall seconds (and fn's dict) there."""
    path = TIMINGS / f"{name}.json"
    if path.exists():
        print(name, "already timed:", path.read_text())
        return
    start = time.perf_counter()
    extra = fn() or {}
    seconds = round(time.perf_counter() - start, 1)
    path.write_text(json.dumps({"seconds": seconds, **extra}, default=str))
    print(name, path.read_text())


def read_timing(name):
    """TIMINGS/<name>.json as a dict, or None when that step has not finished."""
    path = TIMINGS / f"{name}.json"
    return json.loads(path.read_text()) if path.exists() else None


def shapes(workload):
    """The workload's geometries as Step 2 wrote them (EPSG:4326)."""
    return gpd.read_file(SHAPES_OUT / f"{workload}.geojson")


def rise_cubes(workload):
    """id -> datacube.npy of every rise-eo cube of the workload that is on disk."""
    rows = pd.read_csv(OUT / "rise" / workload / "input.csv", dtype={"id": str})
    return {i: Path(p) for i, p in zip(rows["id"], rows["datacube_filepath"]) if Path(p).exists()}


def lacking_files(catalog, bands):
    """Granules of a catalog without one of `bands` (a partly failed download, issue #201)."""
    cat = pd.read_parquet(catalog, columns=["id", "files"])
    return [i for i, f in zip(cat["id"], cat["files"])
            if any(f"{b}.tif" not in f.split(",") for b in bands)]


def mpc_cubes(workload):
    """id -> datacube.npy of every converted mpc-fast-extract cube of the workload."""
    return {p.parent.name: p for p in (OUT / "mpc" / workload / "cubes").glob("*/datacube.npy")}

## Step 1 — check the machine

**What it does:** checks the machine can run mpc-fast-extract and has room, and records what it is.
**PASS if:** Linux x86-64 with AVX2 and FMA; `mpc_fast_extract` is imported from
`MPC_FAST_EXTRACT` at `MPC_FAST_EXTRACT_COMMIT`; `OUT` is not on the network share; and, before
the first download, `OUT` has at least `NEED_GB` free.
**Time / disk:** seconds.

In [ ]:
# Step 1 -- check the machine, and record it.
import platform
import re
import shutil
import subprocess


def git_head(folder):
    """The short commit a checkout is at."""
    return subprocess.run(["git", "rev-parse", "--short=7", "HEAD"], cwd=folder,
                          capture_output=True, text=True, check=True).stdout.strip()


assert platform.system() == "Linux" and platform.machine() == "x86_64", (
    f"this is {platform.system()} {platform.machine()}; mpc-fast-extract needs Linux x86-64")
cpuinfo = Path("/proc/cpuinfo").read_text()
assert re.search(r"\bavx2\b", cpuinfo) and re.search(r"\bfma\b", cpuinfo), (
    "this CPU lacks AVX2 or FMA, which the mpc-fast-extract build targets")
assert Path(mfe.__file__).resolve().is_relative_to(MPC_FAST_EXTRACT.resolve()), (
    f"mpc_fast_extract is imported from {mfe.__file__}, not from {MPC_FAST_EXTRACT}")
assert git_head(MPC_FAST_EXTRACT) == MPC_FAST_EXTRACT_COMMIT, (
    f"mpc-fast-extract is at {git_head(MPC_FAST_EXTRACT)}, not {MPC_FAST_EXTRACT_COMMIT}")
TIMINGS.mkdir(parents=True, exist_ok=True)
assert "cloudfiles" not in str(OUT.resolve()), f"{OUT} is on the network share; pick a local disk"
free_gb = shutil.disk_usage(OUT).free / 1e9
if not ARCHIVE.exists():
    assert free_gb >= NEED_GB, f"only {free_gb:.0f} GB free at {OUT}; want {NEED_GB}"

def physical_cores(cpuinfo):
    """Distinct (physical id, core id) pairs in /proc/cpuinfo, or None when it lists neither."""
    pairs = {(re.search(r"physical id\s*:\s*(\d+)", b), re.search(r"core id\s*:\s*(\d+)", b))
             for b in cpuinfo.strip().split("\n\n")}
    pairs = {(p.group(1), c.group(1)) for p, c in pairs if p and c}
    return len(pairs) or None


machine = {
    "cpu_model": re.search(r"model name\s*:\s*(.+)", cpuinfo).group(1),
    "cpus": os.cpu_count(),
    "physical_cores": physical_cores(cpuinfo),   # hyper-threads share a core's compute
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "rise_eo": git_head(CHECKOUT),
    "mpc_fast_extract": git_head(MPC_FAST_EXTRACT),
}
(TIMINGS / "machine.json").write_text(json.dumps(machine))
print(machine, f"| {free_gb:.0f} GB free at {OUT}")

## Step 2 — write the two workloads' geometries

**What it does:** writes `OUT/shapes/fields.geojson` (the 900 fields, id `field_id`) and
`OUT/shapes/cells.geojson` (`AT_ROI` cut into grid cells by `rise_eo.grid.roi_to_s2_grids`, id
`id`), both in EPSG:4326. Both tools read these same files.
**PASS if:** 900 fields; at least one cell; ids unique in each file; everything lies in UTM 33N
(longitude 12° to 18° E).
**Time / disk:** seconds.

In [ ]:
# Step 2 -- write both workloads' geometries.
from rise_eo.grid import roi_to_s2_grids

SHAPES_OUT.mkdir(parents=True, exist_ok=True)
fields = gpd.read_file(LABELS).to_crs(4326)
fields = fields.assign(field_id=fields["fid"].astype(str))[["field_id", "geometry"]]
cells = roi_to_s2_grids(gpd.read_file(ROI))[["id", "geometry"]]
fields.to_file(SHAPES_OUT / "fields.geojson", driver="GeoJSON")
cells.to_file(SHAPES_OUT / "cells.geojson", driver="GeoJSON")

In [ ]:
# Step 2 -- check it.
for workload, id_col in WORKLOADS.items():
    gdf = shapes(workload)
    assert len(gdf) > 0 and gdf[id_col].is_unique, f"{workload}: ids must be present and unique"
    west, _, east, _ = gdf.total_bounds
    assert 12 <= west and east <= 18, f"{workload}: spans {west:.2f}..{east:.2f} E, outside UTM 33N"
    print(workload, len(gdf), "geometries")
assert len(shapes("fields")) == 900, f"expected 900 fields, found {len(shapes('fields'))}"

## Step 3 — plan both sides (nothing is downloaded)

**What it does:** asks MPC which granules rise-eo would download for each workload, and asks
mpc-fast-extract what it would read (`mpc_fast_extract.plan`: tiles, scenes, blocks, bytes). It
saves both to `OUT/timings/plan.json` and prints mpc-fast-extract's plan tables.
**PASS if:** the granule count is between 1 and `MAX_TILES`, and, before the first download, the
estimated archive size, plus 15 GB for the Zarrs, the cubes and Step 11, fits in the free space at
`OUT`.
**Time / disk:** under a minute.

In [ ]:
# Step 3 -- plan both sides.
from rise_eo.sources import mpc as mpc_source

GB_PER_GRANULE = 0.355  # measured: the first run's 290 granules of B04/B08/SCL took 102.7 GB
plan = {}
for workload in WORKLOADS:
    granules = mpc_source.query_catalog(shapes(workload), STARTDATE, ENDDATE,
                                        collection=COLLECTION, processing=PROCESSING)
    engine_plan = mfe.plan(mpc_config(workload))
    print(f"--- {workload}: mpc-fast-extract plan\n{engine_plan}")
    plan[workload] = {"granules": sorted(granules["id"]),
                      "engine": engine_plan.summary.reset_index().to_dict("records")}
(TIMINGS / "plan.json").write_text(json.dumps(plan, default=str))

In [ ]:
# Step 3 -- check it.
plan = read_timing("plan")
all_granules = set().union(*(plan[w]["granules"] for w in WORKLOADS))
estimate_gb = len(all_granules) * GB_PER_GRANULE
for workload in WORKLOADS:
    print(workload, len(plan[workload]["granules"]), "granules")
print(len(all_granules), f"granules in all, about {estimate_gb:.0f} GB (estimate)")
assert 0 < len(all_granules) <= MAX_TILES, f"{len(all_granules)} granules; MAX_TILES is {MAX_TILES}"
if not ARCHIVE.exists():
    free_gb = shutil.disk_usage(OUT).free / 1e9
    # the Zarrs, the cubes and Step 11 need about 12 GB more after the download
    assert estimate_gb + 15 < free_gb, f"about {estimate_gb:.0f} GB to download (+15), {free_gb:.0f} GB free"

## Step 4 — (a) rise-eo downloads the granules

**What it does:** `rise_eo.download` copies every whole granule both workloads need (B04, B08,
SCL) into one archive at `OUT/archive`, the fields' first, then the cells' (which skips what is
already there). The whole download is timed once; the Summary splits it between the workloads by
granule count.
**PASS if:** the catalog exists, every granule in it has all of `BANDS` (a transfer that
failed leaves its granule without that file, issue #201), and the timing file was written.
If files are missing, run Step 4 again: it fetches only the missing ones.
**Time / disk:** about 7 min on the first run; about 103 GB.

In [ ]:
# Step 4 -- (a) download, timed.
def download_both():
    """Download every granule both workloads need; return the archive's size on disk."""
    for workload in WORKLOADS:
        rise_eo.download(
            roi=shapes(workload), startdate=STARTDATE, enddate=ENDDATE, bands=BANDS,
            dst_folderpath=str(ARCHIVE), source="mpc", collection=COLLECTION,
            processing=PROCESSING, max_tiles=MAX_TILES, max_concurrent=MAX_CONCURRENT)
    size = sum(p.stat().st_size for p in ARCHIVE.rglob("*") if p.is_file())
    return {"archive_gb": round(size / 1e9, 2)}


timed("rise_download", download_both)

In [ ]:
# Step 4 -- check it.
assert Path(CATALOG).exists(), f"no catalog at {CATALOG}"
lacking = lacking_files(CATALOG, BANDS)
assert not lacking, f"{len(lacking)} granules lack a band file; run Step 4 again: {lacking[:3]}"
assert read_timing("rise_download") is not None, "the download did not finish; run Step 4 again"
print(read_timing("rise_download"))

## Step 5 — (b) rise-eo builds the cubes

**What it does:** for each workload, `rise_eo.workflows.create_datacube.run_create_datacube`
builds one cube per geometry from the archive with the local runner, `CORES` builds at a time,
from scratch (`overwrite=True`). Each workload is timed separately.
**PASS if:** every geometry listed in the workload's `input.csv` has its cube on disk. (A geometry
with no granules at all is left out of `input.csv` by rise-eo, and printed.)
**Time / disk:** an estimate of 15–40 min for both; a few GB.

In [ ]:
# Step 5 -- (b) build, timed per workload.
from rise_eo.workflows.create_datacube import run_create_datacube


def rise_build(workload):
    """Build every cube of the workload from scratch with the local runner."""
    run = OUT / "rise" / workload
    run.mkdir(parents=True, exist_ok=True)
    run_create_datacube(
        catalog_filepath=CATALOG, timestamp_col="timestamp",
        shapefilepath=str(SHAPES_OUT / f"{workload}.geojson"), id_col=WORKLOADS[workload],
        run_folderpath=str(run), startdate=STARTDATE, enddate=ENDDATE, bands=BANDS,
        mosaic_days=MOSAIC_DAYS, csv_filepath=str(run / "input.csv"), label_col=None,
        cores=CORES, overwrite=True, collection=COLLECTION, processing=PROCESSING)
    return {"cores": CORES, "cubes": len(rise_cubes(workload))}


for workload in WORKLOADS:
    timed(f"rise_build_{workload}", lambda w=workload: rise_build(w))

In [ ]:
# Step 5 -- check it.
for workload in WORKLOADS:
    rows = pd.read_csv(OUT / "rise" / workload / "input.csv", dtype={"id": str})
    missing = sorted(set(rows["id"]) - set(rise_cubes(workload)))
    assert not missing, f"{workload}: {len(missing)} cubes missing, e.g. {missing[:3]}"
    print(workload, len(rows), "cubes;", read_timing(f"rise_build_{workload}"))

## Step 6 — (c1) mpc-fast-extract composites each workload in one run

**What it does:** one `mpc_fast_extract.run` per workload, over all its geometries at once, with
`mpc_config` from Settings. It reads the imagery straight from MPC (no archive) and writes one
Zarr `(time, band, y, x)` of 20-day medians on the UTM 33N 10 m grid. Each run is timed.
**PASS if:** each run's `issues.json` says the output is complete, and its Zarr exists.
**Time / disk:** an estimate of a few minutes per workload; a few GB.

In [ ]:
# Step 6 -- (c1) one engine run per workload, timed.
def mpc_run(workload):
    """One engine run over all the workload's geometries; return the engine's own summary."""
    run = mfe.run(mpc_config(workload), no_resume=True)
    return {"engine": run.summary, "complete": run.complete}


for workload in WORKLOADS:
    timed(f"mpc_run_{workload}", lambda w=workload: mpc_run(w))

In [ ]:
# Step 6 -- check it.
for workload in WORKLOADS:
    run = mfe.Run(mpc_config(workload), mpc_config(workload)["output"]["path"])
    assert run.complete, f"{workload}: issues.json says the output is incomplete: {run.issues}"
    assert Path(run.zarr_path()).exists(), f"{workload}: no Zarr at {run.zarr_path()}"
    print(workload, read_timing(f"mpc_run_{workload}"))

## Step 7 — (c2) cut the Zarr into rise-eo cubes

**What it does:** for each geometry, cuts its bounding box out of the workload's Zarr and writes
it the way rise-eo lays a cube out: `datacube.npy` `(time, height, width, bands)`, uint16 DN
(reflectance × 10,000), nodata 0, pixels outside the geometry set to 0 (touched pixels kept, as
rise-eo does), every window of `WINDOWS` present; plus `metadata.pickle.npy` with the grid, the
window starts and the band names. This is the work a real backend would add on top of the engine,
so it is timed and counted with it.
**PASS if:** every period in the Zarr is one of rise-eo's windows, and every geometry has a cube.
**Time / disk:** an estimate of a few minutes; a few GB. Memory stays small: it reads one
bounding box at a time, never the whole Zarr.

In [ ]:
# Step 7 -- (c2) convert, timed per workload.
from affine import Affine
from rasterio.crs import CRS
from rasterio.features import geometry_mask

from rise_eo.storage import fs


def window(geom, xs, ys):
    """Row and column slices of the grid pixels whose 10 m cell meets geom's bounding box."""
    minx, miny, maxx, maxy = geom.bounds
    cols = np.flatnonzero((xs + RES / 2 > minx) & (xs - RES / 2 < maxx))
    rows = np.flatnonzero((ys + RES / 2 > miny) & (ys - RES / 2 < maxy))
    return slice(rows[0], rows[-1] + 1), slice(cols[0], cols[-1] + 1)


def save_cube(folder, cube, transform, windows=None, bands=None):
    """Write datacube.npy + metadata.pickle.npy the way rise-eo's builder lays them out."""
    folder.mkdir(parents=True, exist_ok=True)
    np.save(folder / "datacube.npy", cube)
    profile = {"crs": CRS.from_string(GRID_CRS), "transform": transform, "nodata": 0,
               "height": cube.shape[1], "width": cube.shape[2], "count": 1, "dtype": "uint16"}
    meta = {"geotiff_metadata": profile, "bands": bands or CUBE_BANDS,
            "timestamps": list(WINDOWS if windows is None else windows)}
    fs.save_npy(str(folder / "metadata.pickle.npy"), meta, allow_pickle=True)


def mpc_convert(workload):
    """One cube per geometry, reading the run's Zarr one bounding box at a time."""
    run = mfe.Run(mpc_config(workload), mpc_config(workload)["output"]["path"])
    # Lazy, in the engine's 512-px blocks: the Zarr spans whole 100 km work tiles and can be
    # larger than RAM, so it is never loaded whole.
    da = run.zarr(chunks={"time": 1, "band": -1, "y": 512, "x": 512}).sel(band=CUBE_BANDS)
    held = [str(t)[:10] for t in da["time"].values]
    xs, ys = da["x"].values, da["y"].values
    assert ys[0] > ys[-1], "expected a north-up grid"
    gdf = shapes(workload).to_crs(GRID_CRS)
    for cube_id, geom in zip(gdf[WORKLOADS[workload]], gdf.geometry):
        r, c = window(geom, xs, ys)
        part = da.isel(y=r, x=c).reindex(time=WINDOWS).transpose("time", "y", "x", "band").values
        transform = Affine(RES, 0, xs[c.start] - RES / 2, 0, -RES, ys[r.start] + RES / 2)
        cube = np.nan_to_num(part * 1e4, nan=0).round().astype("uint16")
        cube[:, geometry_mask([geom], cube.shape[1:3], transform, all_touched=True), :] = 0
        save_cube(OUT / "mpc" / workload / "cubes" / str(cube_id), cube, transform)
    return {"zarr_periods": held, "zarr_shape": list(da.shape), "cubes": len(gdf)}


for workload in WORKLOADS:
    timed(f"mpc_convert_{workload}", lambda w=workload: mpc_convert(w))

In [ ]:
# Step 7 -- check it.
expected = [str(w.date()) for w in WINDOWS]
for workload in WORKLOADS:
    held = read_timing(f"mpc_convert_{workload}")["zarr_periods"]
    extra = sorted(set(held) - set(expected))
    assert not extra, f"{workload}: the Zarr holds periods {extra} that are not rise-eo windows {expected}"
    assert len(mpc_cubes(workload)) == len(shapes(workload)), f"{workload}: some cubes are missing"
    print(workload, len(mpc_cubes(workload)), "cubes; windows without data:",
          sorted(set(expected) - set(held)))

## Step 8 — compare the two cubes of every geometry

**What it does:** for each geometry with both cubes, lines them up by their transforms (they must
share one 10 m pixel lattice) and compares the pixels both cover, over every window and band. It
counts: same shape and grid, same window dates, pixels where exactly one side is nodata, and,
where both have a value, pixels that are equal and pixels within 1 DN (a median of an even count
can round either way). Saved to `OUT/compare/<workload>.json`.
The counts are split by how many MGRS tiles fed the rise-eo cube. Where two tiles overlap, one
overpass comes as two products whose values differ by a few DN, and the two tools keep different
ones (rise-eo the first in catalog order, mpc-fast-extract the first by datetime, then id). Both
are valid observations, so the multi-tile counts are reported, not checked.
**PASS if,** for each workload:
- no geometry has a rise-eo cube without an mpc-fast-extract cube;
- every compared pair has the same window dates;
- on single-tile geometries, nodata agrees on at least `MIN_AGREEMENT` of the pixels, and values
  agree within 1 DN on at least `MIN_AGREEMENT` of the pixels where both have one.

`MIN_AGREEMENT` is a first guess. A miss is a finding to look at, not a broken run.
**Time / disk:** a few minutes.

In [ ]:
# Step 8 -- the comparison helpers.
def load_cube(path, bands=None):
    """(cube with `bands` (default CUBE_BANDS) in that order, its transform, CRS, window dates)."""
    meta = fs.load_npy(str(path.parent / "metadata.pickle.npy"), allow_pickle=True)[()]
    cube = np.load(path)[..., [list(meta["bands"]).index(b) for b in bands or CUBE_BANDS]]
    dates = [str(pd.Timestamp(t).date()) for t in meta["timestamps"]]
    profile = meta["geotiff_metadata"]
    return cube, profile["transform"], CRS.from_user_input(profile["crs"]), dates


def overlap(a, ta, b, tb):
    """The parts of cubes a and b (t, H, W, bands) over the same pixels, found by their transforms."""
    dc, dr = (tb.c - ta.c) / RES, (ta.f - tb.f) / RES
    assert abs(dc - round(dc)) < 1e-6 and abs(dr - round(dr)) < 1e-6, "not one pixel lattice"
    dc, dr = round(dc), round(dr)
    r0, c0 = max(0, dr), max(0, dc)
    r1, c1 = min(a.shape[1], dr + b.shape[1]), min(a.shape[2], dc + b.shape[2])
    return a[:, r0:r1, c0:c1], b[:, r0 - dr:r1 - dr, c0 - dc:c1 - dc]


def tile_count(cube_path):
    """How many MGRS tiles fed a rise-eo cube, from the catalog slice next to it."""
    ids = pd.read_parquet(cube_path.parent / "catalog.parquet", columns=["id"])["id"]
    return ids.str.extract(r"_(T\d{2}[A-Z]{3})(?:_|$)")[0].nunique()


def compare(workload):
    """Agreement counts for single- and multi-tile geometries, saved to OUT/compare/<workload>.json."""
    rise, mpc = rise_cubes(workload), mpc_cubes(workload)
    keys = ["cubes", "same_grid", "same_windows", "pixels", "one_nodata", "both_valid", "equal",
            "within_1"]
    n = {"only_rise": len(set(rise) - set(mpc)), "only_mpc": len(set(mpc) - set(rise)),
         "single_tile": dict.fromkeys(keys, 0), "multi_tile": dict.fromkeys(keys, 0)}
    for cube_id in sorted(set(rise) & set(mpc)):
        a, ta, crs, wa = load_cube(rise[cube_id])
        b, tb, _, wb = load_cube(mpc[cube_id])
        assert crs == CRS.from_string(GRID_CRS), f"{cube_id}: rise-eo built it in {crs}"
        g = n["single_tile" if tile_count(rise[cube_id]) == 1 else "multi_tile"]
        g["cubes"] += 1
        g["same_grid"] += a.shape == b.shape and ta == tb
        g["same_windows"] += wa == wb
        a, b = overlap(a, ta, b, tb)
        both, diff = (a > 0) & (b > 0), np.abs(a.astype(np.int32) - b)
        g["pixels"] += a.size
        g["one_nodata"] += int(((a > 0) != (b > 0)).sum())
        g["both_valid"] += int(both.sum())
        g["equal"] += int((diff[both] == 0).sum())
        g["within_1"] += int((diff[both] <= 1).sum())
    (OUT / "compare").mkdir(exist_ok=True)
    (OUT / "compare" / f"{workload}.json").write_text(json.dumps(n))
    return n


for workload in WORKLOADS:
    print(workload, compare(workload))

In [ ]:
# Step 8 -- check it.
def shares(g):
    """(nodata agreement, share within 1 DN, share equal) of one group's counts."""
    return (1 - g["one_nodata"] / g["pixels"], g["within_1"] / g["both_valid"],
            g["equal"] / g["both_valid"])


for workload in WORKLOADS:
    n = json.loads((OUT / "compare" / f"{workload}.json").read_text())
    assert n["only_rise"] == 0, f"{workload}: {n['only_rise']} rise-eo cubes have no counterpart"
    for group in ("single_tile", "multi_tile"):
        g = n[group]
        if g["cubes"]:
            nodata_agree, within_1, equal = shares(g)
            print(f"{workload} {group} ({g['cubes']} cubes): nodata agrees on {nodata_agree:.4%}, "
                  f"values within 1 DN on {within_1:.4%}, equal on {equal:.4%}")
        assert g["same_windows"] == g["cubes"], f"{workload} {group}: window dates differ"
    nodata_agree, within_1, _ = shares(n["single_tile"])
    assert nodata_agree >= MIN_AGREEMENT, f"{workload}: single-tile nodata agrees on {nodata_agree:.4%}"
    assert within_1 >= MIN_AGREEMENT, f"{workload}: single-tile values within 1 DN on {within_1:.4%}"

## Step 9 — (b, fair) rise-eo builds again, one build per core

**What it does:** Step 5 ran `CORES` builds at a time, half the cores, while mpc-fast-extract
used them all. This rebuilds every cube from scratch with `CORES_FAIR` builds at a time (one per
core) and times each workload again, as `rise_build_fair_<workload>`; Step 5's times are kept.
It also turns on rise-eo's per-cube timing record (`timings.json` next to each cube: seconds per
build phase), which Step 10 exports for the report's phase breakdown. The cubes come out the
same, since the code and inputs are the same.
**PASS if:** every cube is on disk again with its `timings.json`.
If this step ever reports cubes missing, the timing is not valid: delete
`OUT/timings/rise_build_fair_*.json` and run it again.
**Time / disk:** an estimate of 20–45 min for both; nothing new on disk beyond the small
`timings.json` files. If the kernel dies here, memory ran out: lower `CORES_FAIR` by one in
Settings and run this step again.

In [ ]:
# Step 9 -- (b, fair) rebuild with one build per core, per-cube timings on, timed per workload.
import shutil
import tempfile

from rise_eo.workflows.create_datacube import run_create_datacube

# rise-eo's own switch for each cube's timings.json. The run-book rules say no environment
# variables; this is the one exception, because the builder has no argument for it here.
os.environ["RISE_EO_WRITE_TIMINGS"] = "1"


def rise_build_fair(workload):
    """Rebuild every cube of the workload from scratch, CORES_FAIR builds at a time."""
    # The local runner's done.txt sentinels outlive the run that wrote them, so a forced
    # rebuild would delete every cube and build none (issue #200). Clear them first.
    shutil.rmtree(Path(tempfile.gettempdir()) / "rise-eo-snakemake-scratch", ignore_errors=True)
    run = OUT / "rise" / workload
    run_create_datacube(
        catalog_filepath=CATALOG, timestamp_col="timestamp",
        shapefilepath=str(SHAPES_OUT / f"{workload}.geojson"), id_col=WORKLOADS[workload],
        run_folderpath=str(run), startdate=STARTDATE, enddate=ENDDATE, bands=BANDS,
        mosaic_days=MOSAIC_DAYS, csv_filepath=str(run / "input.csv"), label_col=None,
        cores=CORES_FAIR, overwrite=True, collection=COLLECTION, processing=PROCESSING)
    return {"cores": CORES_FAIR, "cubes": len(rise_cubes(workload))}


for workload in WORKLOADS:
    timed(f"rise_build_fair_{workload}", lambda w=workload: rise_build_fair(w))

In [ ]:
# Step 9 -- check it.
for workload in WORKLOADS:
    cubes = rise_cubes(workload)
    rows = pd.read_csv(OUT / "rise" / workload / "input.csv", dtype={"id": str})
    assert len(cubes) == len(rows), f"{workload}: {len(rows) - len(cubes)} cubes missing"
    untimed = [i for i, p in cubes.items() if not (p.parent / "timings.json").exists()]
    assert not untimed, f"{workload}: {len(untimed)} cubes have no timings.json, e.g. {untimed[:3]}"
    print(workload, read_timing(f"rise_build_fair_{workload}"),
          "| Step 5 took", read_timing(f"rise_build_{workload}")["seconds"], "s")

## Step 10 — export what the report's figures need

**What it does:** writes small files into `EXPORT` (under this checkout's `tests/outputs/`, on the
home disk, so they survive stopping the VM):
- `timings/`, `compare/` and both workloads' geometries, copied as they are;
- `rise_phases_<workload>.csv`: each rise-eo cube's `timings.json` phases and its shape;
- `plan_tiles_<workload>.json`: mpc-fast-extract's plan per MGRS tile (blocks it needs), which asks
  MPC for the scene list again;
- `tile_grids.json`: each MGRS tile's 10 m grid, read from one B08 file per tile in the archive;
- for each workload, from every pair of cubes (both cubes on the rise-eo cube's grid):
  `per_geometry_<workload>.csv` (counts and the tile count per geometry),
  `hist_<workload>.csv` (how often each difference, rise-eo minus mpc-fast-extract, from −300 to
  300 DN, occurs, single- vs multi-tile), `window_band_<workload>.csv` (pixels both valid and pixels
  off by more than 1 DN, per window and band), and `<workload>_disagreement.tif`, one 10 m raster
  on UTM 33N whose four bands count, per pixel and over all windows × bands: the slots off by more
  than 1 DN, the largest difference, the slots where one side has no value, and the slots both
  have. Where cells overlap, a pixel keeps the larger count.

It uses `load_cube` and `tile_count` from Step 8's helper cell: after a restart, run that cell
first.
**PASS if:** every file listed above exists.
**Time / disk:** an estimate of 5–10 min; well under 200 MB.

**Then copy `EXPORT` to your Mac:** in VS Code's Explorer, right-click
`tests/outputs/198_p0_export` → **Download…**, and save it as `tests/outputs/198_p0_export` in your
Mac's rise-eo checkout. After that the VM can be stopped.

In [ ]:
# Step 10 -- the export helpers.
import re
import shutil

import rasterio
from affine import Affine
from rasterio.crs import CRS

from rise_eo.storage import fs


def on_grid_of(a, ta, b, tb):
    """Cube b (t, H, W, bands) re-cut onto cube a's pixel grid; 0 where b has no pixel."""
    out = np.zeros(a.shape, dtype=b.dtype)
    dc, dr = round((tb.c - ta.c) / RES), round((ta.f - tb.f) / RES)
    r0, c0 = max(0, dr), max(0, dc)
    r1, c1 = min(a.shape[1], dr + b.shape[1]), min(a.shape[2], dc + b.shape[2])
    out[:, r0:r1, c0:c1] = b[:, r0 - dr:r1 - dr, c0 - dc:c1 - dc]
    return out


def rise_phases(workload):
    """One row per rise-eo cube: its timings.json phases, total seconds and cube shape."""
    rows = []
    for cube_id, path in rise_cubes(workload).items():
        t = json.loads((path.parent / "timings.json").read_text())
        T, H, W, _ = np.load(path, mmap_mode="r").shape
        rows.append({"id": cube_id, "total_seconds": t["total_seconds"], "T": T, "H": H, "W": W,
                     **{f"phase_{k}": v for k, v in t["phase_seconds"].items()}})
    return pd.DataFrame(rows)


def tile_grids():
    """MGRS tile -> CRS, transform, width and height of its 10 m grid, from one B08 file each."""
    grids = {}
    for path in sorted(ARCHIVE.rglob("*B08*.tif")):
        tile = re.search(r"_(T\d{2}[A-Z]{3})_", str(path))
        if tile and tile.group(1) not in grids:
            with rasterio.open(path) as src:
                grids[tile.group(1)] = {"crs": str(src.crs), "transform": list(src.transform)[:6],
                                        "width": src.width, "height": src.height}
    return grids


LAYERS = ["slots_off_by_more_than_1_dn", "max_abs_diff_dn", "slots_one_side_nodata",
          "slots_both_valid"]


def write_map(workload, maps, transform):
    """The four per-pixel layers as one GeoTIFF for QGIS."""
    profile = dict(driver="GTiff", height=maps.shape[1], width=maps.shape[2], count=len(LAYERS),
                   dtype="uint16", crs=GRID_CRS, transform=transform, compress="deflate",
                   tiled=True, blockxsize=512, blockysize=512)
    with rasterio.open(EXPORT / f"{workload}_disagreement.tif", "w", **profile) as dst:
        dst.write(maps)
        for k, name in enumerate(LAYERS, start=1):
            dst.set_band_description(k, name)


def export_pixels(workload):
    """Per-geometry, per-difference, per-window and per-pixel disagreement of one workload."""
    rise, mpc = rise_cubes(workload), mpc_cubes(workload)
    minx, miny, maxx, maxy = shapes(workload).to_crs(GRID_CRS).total_bounds
    x0, y1 = math.floor(minx / RES) * RES, math.ceil(maxy / RES) * RES
    maps = np.zeros((len(LAYERS), math.ceil((y1 - miny) / RES) + 1,
                     math.ceil((maxx - x0) / RES) + 1), dtype="uint16")
    hist = {g: np.zeros(601, dtype="int64") for g in ("single_tile", "multi_tile")}
    windows = {g: np.zeros((len(WINDOWS), len(CUBE_BANDS), 2), dtype="int64") for g in hist}
    rows = []
    for cube_id in sorted(set(rise) & set(mpc)):
        a, ta, _, _ = load_cube(rise[cube_id])
        b, tb, _, _ = load_cube(mpc[cube_id])
        b = on_grid_of(a, ta, b, tb)
        d = a.astype(np.int32) - b
        both, one = (a > 0) & (b > 0), (a > 0) != (b > 0)
        off, tiles = both & (np.abs(d) > 1), tile_count(rise[cube_id])
        group = "single_tile" if tiles == 1 else "multi_tile"
        hist[group] += np.bincount(np.clip(d[both], -300, 300) + 300, minlength=601)
        windows[group] += np.stack([both.sum(axis=(1, 2)), off.sum(axis=(1, 2))], axis=-1)
        r, c = round((y1 - ta.f) / RES), round((ta.c - x0) / RES)
        layers = [off.sum(axis=(0, 3)), np.where(both, np.abs(d), 0).max(axis=(0, 3)),
                  one.sum(axis=(0, 3)), both.sum(axis=(0, 3))]
        for k, layer in enumerate(layers):
            win = maps[k, r:r + a.shape[1], c:c + a.shape[2]]
            np.maximum(win, layer.astype("uint16"), out=win)
        rows.append({"id": cube_id, "tiles": tiles, "both_valid": int(both.sum()),
                     "off": int(off.sum()), "equal": int((both & (d == 0)).sum()),
                     "one_nodata": int(one.sum()), "max_abs_diff": int(np.abs(d[both]).max(initial=0))})
    pd.DataFrame(rows).to_csv(EXPORT / f"per_geometry_{workload}.csv", index=False)
    pd.DataFrame({"diff": np.arange(-300, 301), **hist}).to_csv(EXPORT / f"hist_{workload}.csv", index=False)
    pd.DataFrame([{"window": str(w.date()), "band": band, "group": g, "both_valid": int(v[t, k, 0]),
                   "off": int(v[t, k, 1])} for g, v in windows.items()
                  for t, w in enumerate(WINDOWS) for k, band in enumerate(CUBE_BANDS)]
                 ).to_csv(EXPORT / f"window_band_{workload}.csv", index=False)
    write_map(workload, maps, Affine(RES, 0, x0, 0, -RES, y1))

In [ ]:
# Step 10 -- export.
EXPORT.mkdir(parents=True, exist_ok=True)
shutil.copytree(TIMINGS, EXPORT / "timings", dirs_exist_ok=True)
shutil.copytree(OUT / "compare", EXPORT / "compare", dirs_exist_ok=True)
(EXPORT / "tile_grids.json").write_text(json.dumps(tile_grids()))
for workload in WORKLOADS:
    shutil.copy(SHAPES_OUT / f"{workload}.geojson", EXPORT / f"{workload}.geojson")
    rise_phases(workload).to_csv(EXPORT / f"rise_phases_{workload}.csv", index=False)
    mfe.plan(mpc_config(workload)).tiles.to_json(EXPORT / f"plan_tiles_{workload}.json",
                                                 orient="records")
    export_pixels(workload)
    print(workload, "exported")

In [ ]:
# Step 10 -- check it.
expected = ["tile_grids.json", "timings", "compare"] + [
    name.format(w) for w in WORKLOADS for name in (
        "{}.geojson", "rise_phases_{}.csv", "plan_tiles_{}.json", "per_geometry_{}.csv",
        "hist_{}.csv", "window_band_{}.csv", "{}_disagreement.tif")]
missing = [name for name in expected if not (EXPORT / name).exists()]
assert not missing, f"missing from {EXPORT}: {missing}"
assert len(json.loads((EXPORT / "tile_grids.json").read_text())) >= 4, "fewer than 4 MGRS tiles found"
size_mb = sum(p.stat().st_size for p in EXPORT.rglob("*") if p.is_file()) / 1e6
print(f"{len(expected)} items, {size_mb:.0f} MB in {EXPORT}")

## Step 11 — the radiometric offset and a 20 m band, on 2023 imagery

**What it does:** the 2018 imagery above carries no radiometric offset, so the −1000 DN offset of
processing baseline 04.00 and later was never exercised, and B04/B08 are both 10 m bands, so
neither was resampling. This runs both tools once more, on the single-tile cell `OFFSET_ROI`,
June 2023, 10-day windows, with B04, B08 and the 20 m band B8A:
1. rise-eo downloads the granules into `OFFSET_RUN/archive` and builds the one cube;
2. mpc-fast-extract composites it with the same config as Step 6 (dates, bands and paths changed);
3. Step 7's cut turns its Zarr into a rise-eo cube; the two cubes are compared per band.
The result is saved to `OUT/compare/offset_2023.json` and into `EXPORT`.

It uses `window`, `save_cube` (Step 7) and `load_cube`, `overlap` (Step 8): after a restart, run
those two helper cells first.
**PASS if:**
- every downloaded granule has all of `OFFSET_BANDS` (issue #201), offset −1000 and processing
  baseline 04.00 or later (so the offset path really is exercised), and all come from one MGRS
  tile;
- the window dates match;
- for each band, nodata agrees on at least `MIN_AGREEMENT` of the pixels, and values agree within
  1 DN on at least `MIN_AGREEMENT` of the pixels both have.

**Time / disk:** an estimate of 5–10 min; about 5 GB.

In [ ]:
# Step 11 -- the offset check: download, build, composite, cut.
import shutil
import tempfile

from rise_eo.workflows.create_datacube import run_create_datacube

OFFSET_CATALOG = archive_catalog_filepath(str(OFFSET_RUN / "archive"), COLLECTION)


def offset_config():
    """mpc_config with the offset check's geometry, dates, bands and output folder."""
    cfg = mpc_config("cells")
    cfg.update(geometries=str(OFFSET_RUN / "roi.geojson"), id_field="id",
               start=OFFSET_START.date().isoformat(), period=f"{OFFSET_MOSAIC_DAYS}d",
               end=(OFFSET_END - datetime.timedelta(days=1)).date().isoformat())
    cfg["sentinel2"]["bands"] = OFFSET_CUBE_BANDS
    cfg["output"]["path"] = str(OFFSET_RUN / "mpc_engine")
    return cfg


def offset_rise():
    """Download the cell's granules and build its rise-eo cube."""
    rise_eo.download(roi=gpd.read_file(OFFSET_RUN / "roi.geojson"), startdate=OFFSET_START,
                     enddate=OFFSET_END, bands=OFFSET_BANDS, dst_folderpath=str(OFFSET_RUN / "archive"),
                     source="mpc", collection=COLLECTION, processing=PROCESSING,
                     max_tiles=OFFSET_MAX_TILES, max_concurrent=MAX_CONCURRENT)
    shutil.rmtree(Path(tempfile.gettempdir()) / "rise-eo-snakemake-scratch", ignore_errors=True)  # #200
    run_create_datacube(
        catalog_filepath=OFFSET_CATALOG, timestamp_col="timestamp",
        shapefilepath=str(OFFSET_RUN / "roi.geojson"), id_col="id",
        run_folderpath=str(OFFSET_RUN / "rise"), startdate=OFFSET_START, enddate=OFFSET_END,
        bands=OFFSET_BANDS, mosaic_days=OFFSET_MOSAIC_DAYS,
        csv_filepath=str(OFFSET_RUN / "rise" / "input.csv"), label_col=None, cores=1,
        overwrite=True, collection=COLLECTION, processing=PROCESSING)


def offset_mpc():
    """Composite the cell with mpc-fast-extract and cut its Zarr into one rise-eo-shaped cube."""
    cfg = offset_config()
    run = mfe.run(cfg, no_resume=True)
    assert run.complete, f"issues.json says the output is incomplete: {run.issues}"
    da = run.zarr(chunks={"time": 1, "band": -1, "y": 512, "x": 512}).sel(band=OFFSET_CUBE_BANDS)
    xs, ys = da["x"].values, da["y"].values
    geom = gpd.read_file(OFFSET_RUN / "roi.geojson").to_crs(GRID_CRS).geometry.iloc[0]
    r, c = window(geom, xs, ys)
    part = da.isel(y=r, x=c).reindex(time=OFFSET_WINDOWS).transpose("time", "y", "x", "band").values
    transform = Affine(RES, 0, xs[c.start] - RES / 2, 0, -RES, ys[r.start] + RES / 2)
    cube = np.nan_to_num(part * 1e4, nan=0).round().astype("uint16")
    cube[:, geometry_mask([geom], cube.shape[1:3], transform, all_touched=True), :] = 0
    save_cube(OFFSET_RUN / "mpc", cube, transform, windows=OFFSET_WINDOWS, bands=OFFSET_CUBE_BANDS)
    return {"engine": run.summary, "zarr_periods": [str(t)[:10] for t in da["time"].values]}


(OFFSET_RUN / "rise").mkdir(parents=True, exist_ok=True)
gpd.read_file(OFFSET_ROI)[["id", "geometry"]].to_crs(4326).to_file(OFFSET_RUN / "roi.geojson",
                                                                  driver="GeoJSON")
timed("offset_rise", offset_rise)
timed("offset_mpc", offset_mpc)

In [ ]:
# Step 11 -- compare the two cubes per band, and save the result.
def offset_compare():
    """Per-band agreement of the offset check's two cubes, plus what its catalog says."""
    granules = pd.read_parquet(OFFSET_CATALOG, columns=["id", "offset", "processing_version"])
    rise_path = Path(pd.read_csv(OFFSET_RUN / "rise" / "input.csv")["datacube_filepath"].iloc[0])
    a, ta, _, wa = load_cube(rise_path, bands=OFFSET_CUBE_BANDS)
    b, tb, _, wb = load_cube(OFFSET_RUN / "mpc" / "datacube.npy", bands=OFFSET_CUBE_BANDS)
    a, b = overlap(a, ta, b, tb)
    d = a.astype(np.int32) - b
    bands = {}
    for k, band in enumerate(OFFSET_CUBE_BANDS):
        ak, bk, dk = a[..., k], b[..., k], d[..., k]
        both = (ak > 0) & (bk > 0)
        bands[band] = {"pixels": int(ak.size), "one_nodata": int(((ak > 0) != (bk > 0)).sum()),
                       "both_valid": int(both.sum()), "equal": int((dk[both] == 0).sum()),
                       "within_1": int((np.abs(dk[both]) <= 1).sum()),
                       "diff_percentiles_1_50_99": np.percentile(dk[both], [1, 50, 99]).tolist()}
    tiles = granules["id"].str.extract(r"_(T\d{2}[A-Z]{3})(?:_|$)")[0].unique().tolist()
    return {"granules": len(granules), "tiles": tiles,
            "lacking_a_band_file": lacking_files(OFFSET_CATALOG, OFFSET_BANDS), "offsets": sorted(granules["offset"].unique().tolist()),
            "processing_versions": sorted(map(str, granules["processing_version"].unique())),
            "same_windows": wa == wb, "same_grid": a.shape == b.shape, "bands": bands}


result = offset_compare()
for folder in (OUT / "compare", EXPORT / "compare"):
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "offset_2023.json").write_text(json.dumps(result))
print(json.dumps(result, indent=1))

In [ ]:
# Step 11 -- check it.
result = json.loads((OUT / "compare" / "offset_2023.json").read_text())
assert not result["lacking_a_band_file"], (
    f"granules without a band file: {result['lacking_a_band_file']}. Delete "
    "OUT/timings/offset_rise.json and run Step 11 again: the download fetches only the missing files")
assert result["offsets"] == [-1000], f"granule offsets are {result['offsets']}, not all -1000"
assert all(v != "None" and float(v) >= 4 for v in result["processing_versions"]), (
    f"processing baselines {result['processing_versions']}: not all 04.00 or later")
assert len(result["tiles"]) == 1, f"the cell's granules come from {result['tiles']}, not one tile"
assert result["same_windows"], "window dates differ"
for band, n in result["bands"].items():
    nodata_agree, within_1 = 1 - n["one_nodata"] / n["pixels"], n["within_1"] / n["both_valid"]
    print(f"{band}: nodata agrees on {nodata_agree:.4%}, within 1 DN on {within_1:.4%}, "
          f"equal on {n['equal'] / n['both_valid']:.4%}")
    assert nodata_agree >= MIN_AGREEMENT, f"{band}: nodata agrees on only {nodata_agree:.4%}"
    assert within_1 >= MIN_AGREEMENT, f"{band}: values within 1 DN on only {within_1:.4%}"

## Summary — paste this output back

Re-reads everything from disk, so it works after a restart (run Setup, Settings and Helpers first),
and after a failed check. A step that has not finished shows `None`.

- `rise_download_s_est`: the download time split by this workload's share of the granules (an
  estimate: the two workloads were downloaded into one archive).
- `ratio_build_only`: rise-eo's build time ÷ mpc-fast-extract's (run + conversion), as if the
  archive were already on disk.
- `ratio_with_download`: the same, counting rise-eo's estimated download time too.

In [ ]:
import json


def seconds(name):
    """A timed step's wall seconds, or None when it has not finished."""
    t = read_timing(name)
    return t and t["seconds"]


def ratio(a, b):
    """a / b rounded, or None when either is missing."""
    return round(a / b, 1) if a and b else None


plan = read_timing("plan") or {}
all_granules = set().union(*(plan.get(w, {}).get("granules", []) for w in WORKLOADS))
summary = {"machine": read_timing("machine"), "cores_for_rise_builds": CORES,
           "cores_for_fair_rise_builds": CORES_FAIR,
           "rise_download": read_timing("rise_download")}
for workload in WORKLOADS:
    granules = plan.get(workload, {}).get("granules", [])
    share = len(granules) / len(all_granules) if all_granules else None
    download = seconds("rise_download") and share and round(seconds("rise_download") * share, 1)
    build = seconds(f"rise_build_{workload}")
    fair = seconds(f"rise_build_fair_{workload}")
    run, convert = seconds(f"mpc_run_{workload}"), seconds(f"mpc_convert_{workload}")
    mpc_total = run and convert and run + convert
    compared = OUT / "compare" / f"{workload}.json"
    summary[workload] = {
        "granules": len(granules),
        "rise_download_s_est": download, "rise_build_s": build,
        "mpc_run_s": run, "mpc_convert_s": convert,
        "ratio_build_only": ratio(build, mpc_total),
        "ratio_with_download": ratio(download and build and download + build, mpc_total),
        "rise_build_fair_s": fair, "ratio_fair_build_only": ratio(fair, mpc_total),
        "ratio_fair_with_download": ratio(download and fair and download + fair, mpc_total),
        "mpc_engine": (read_timing(f"mpc_run_{workload}") or {}).get("engine"),
        "compare": json.loads(compared.read_text()) if compared.exists() else None,
    }
offset = OUT / "compare" / "offset_2023.json"
summary["offset_2023"] = json.loads(offset.read_text()) if offset.exists() else None
print(json.dumps(summary, indent=2, default=str))